In [1]:
import pandas as pd

# ========== 1. 读取原始文件 ==========
input_file = "1819_no_wr_no_weather.csv"
df = pd.read_csv(input_file)

# ========== 2. 按 UID 去重（保留第一行） ==========
df_dedup = df.drop_duplicates(
    subset=["UID"],
    keep="first"
)

# ========== 3. 输出基本信息 ==========
print("原始行数:", df.shape[0])
print("去重后行数:", df_dedup.shape[0])
print("去掉的重复行数:", df.shape[0] - df_dedup.shape[0])

# ========== 4. 保存新文件 ==========
output_file = "1314_no_wr_no_weather_uid_dedup.csv"
df_dedup.to_csv(output_file, index=False)

print("\n完成 ✔")
print("去重后的文件已保存为:", output_file)

原始行数: 87079
去重后行数: 2106
去掉的重复行数: 84973

完成 ✔
去重后的文件已保存为: 1314_no_wr_no_weather_uid_dedup.csv


In [12]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt

# =====================================================
# 1. 文件路径
# =====================================================
PATH_1819 = "1819_no_wr_no_weather_uid_dedup.csv"
PATH_STATIONS = "ghcnd-stations.txt"
OUTPUT_PATH = "1819_with_nearest_station.csv"

# =====================================================
# 2. 读取 1819 数据
# ⚠️ 如果你列名不是 LAT / LON，在这里改
# =====================================================
df_1819 = pd.read_csv(PATH_1819)

required_cols = ["UID", "LAT_DD83", "LON_DD83"]
for col in required_cols:
    if col not in df_1819.columns:
        raise ValueError(f"❌ 缺少必要列: {col}")

print(f"✅ 1819 数据读取成功，共 {len(df_1819)} 条")

# =====================================================
# 3. 读取 GHCN 站点（严格按 NOAA 官方定宽）
# =====================================================
stations = pd.read_fwf(
    PATH_STATIONS,
    colspecs=[
        (0, 11),    # ID
        (12, 20),   # LATITUDE
        (21, 30),   # LONGITUDE
        (31, 37),   # ELEVATION
        (38, 40),   # STATE
        (41, 71),   # NAME
        (72, 75),   # GSN FLAG
        (76, 79),   # HCN/CRN FLAG
        (80, 85)    # WMO ID
    ],
    names=[
        "station_id",
        "latitude",
        "longitude",
        "elevation",
        "state",
        "name",
        "gsn_flag",
        "hcn_crn_flag",
        "wmo_id"
    ]
)

stations = stations.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

print(f"✅ 气象站点读取成功，共 {len(stations)} 个站点")

# =====================================================
# 4. Haversine 球面距离函数（km）
# =====================================================
def haversine(lon1, lat1, lon2, lat2):
    """
    lon1, lat1: 标量
    lon2, lat2: numpy array
    返回：每个站点到该点的距离（km）
    """
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c  # km

# =====================================================
# 5. 查找最近气象站
# =====================================================
def find_nearest_station(row, stations_df):
    distances = haversine(
        row["LON_DD83"],
        row["LAT_DD83"],
        stations_df["longitude"].values,
        stations_df["latitude"].values
    )

    idx = np.argmin(distances)

    return pd.Series({
        "nearest_station_id": stations_df.iloc[idx]["station_id"],
        "nearest_station_lat": stations_df.iloc[idx]["latitude"],
        "nearest_station_lon": stations_df.iloc[idx]["longitude"],
        "nearest_station_elevation": stations_df.iloc[idx]["elevation"],
        "nearest_station_state": stations_df.iloc[idx]["state"],
        "station_distance_km": distances[idx]
    })

print("🚀 开始匹配最近气象站（可能需要几分钟）...")

nearest_station_info = df_1819.apply(
    find_nearest_station,
    axis=1,
    stations_df=stations
)

# =====================================================
# 6. 合并并保存
# =====================================================
df_final = pd.concat([df_1819, nearest_station_info], axis=1)

df_final.to_csv(OUTPUT_PATH, index=False)

print("🎉 完成！结果已保存：", OUTPUT_PATH)
print(df_final.head())

✅ 1819 数据读取成功，共 2106 条
✅ 气象站点读取成功，共 129657 个站点
🚀 开始匹配最近气象站（可能需要几分钟）...
🎉 完成！结果已保存： 1819_with_nearest_station.csv
       UID      PHYLUM         CLASS           ORDER           FAMILY  \
0  2014737  ARTHROPODA  MALACOSTRACA        DECAPODA       CAMBARIDAE   
1  2012488  ARTHROPODA       INSECTA         DIPTERA        TABANIDAE   
2  2012489  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
3  2012490  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
4  2012491  ARTHROPODA     ARACHNIDA  TROMBIDIFORMES  TORRENTICOLIDAE   

          GENUS  TARGET_TAXON  TAXA_ID  TOTAL300  IS_DISTINCT300  ...  \
0      FAXONIUS      FAXONIUS     5131       1.0             1.0  ...   
1           NaN     TABANIDAE     4340       1.0             1.0  ...   
2           NaN  CHIRONOMIDAE     3581       2.0             0.0  ...   
3           NaN  CHIRONOMIDAE     3581       NaN             NaN  ...   
4  TORRENTICOLA  TORRENTICOLA     4371       NaN             NaN  ...   

   VISIT_

In [17]:
import pandas as pd
import numpy as np

# =====================================================
# 1. 文件路径
# =====================================================
PATH_1819 = "1819_no_wr_no_weather_uid_dedup.csv"
PATH_STATIONS = "ghcnd-stations.txt"
OUTPUT_PATH = "1819_with_10_nearest_stations.csv"

# =====================================================
# 2. 读取 1819 数据
# =====================================================
df_1819 = pd.read_csv(PATH_1819)

required_cols = ["UID", "LAT_DD83", "LON_DD83"]
for col in required_cols:
    if col not in df_1819.columns:
        raise ValueError(f"❌ 缺少必要列: {col}")

print(f"✅ 1819 数据读取成功，共 {len(df_1819)} 条")

# =====================================================
# 3. 读取 GHCN 站点
# =====================================================
stations = pd.read_fwf(
    PATH_STATIONS,
    colspecs=[
        (0, 11),    # ID
        (12, 20),   # LATITUDE
        (21, 30),   # LONGITUDE
        (31, 37),   # ELEVATION
        (38, 40),   # STATE
        (41, 71),   # NAME
        (72, 75),   # GSN FLAG
        (76, 79),   # HCN/CRN FLAG
        (80, 85)    # WMO ID
    ],
    names=[
        "station_id",
        "latitude",
        "longitude",
        "elevation",
        "state",
        "name",
        "gsn_flag",
        "hcn_crn_flag",
        "wmo_id"
    ]
)

stations = stations.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
print(f"✅ 气象站点读取成功，共 {len(stations)} 个站点")

# =====================================================
# 4. Haversine 距离函数（km）
# =====================================================
def haversine(lon1, lat1, lon2, lat2):
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c  # km

# =====================================================
# 5. 查找最近的 5 个气象站（核心改动）
# =====================================================
def find_k_nearest_stations(row, stations_df, k=10):
    distances = haversine(
        row["LON_DD83"],
        row["LAT_DD83"],
        stations_df["longitude"].values,
        stations_df["latitude"].values
    )

    nearest_idx = np.argsort(distances)[:k]

    result = {}

    for i, idx in enumerate(nearest_idx, start=1):
        result[f"candidate_station_{i}_id"] = stations_df.iloc[idx]["station_id"]
        result[f"candidate_station_{i}_distance_km"] = distances[idx]

    return pd.Series(result)

print("🚀 开始匹配最近的 5 个气象站（可能需要几分钟）...")

nearest_stations_info = df_1819.apply(
    find_k_nearest_stations,
    axis=1,
    stations_df=stations,
    k=5
)

# =====================================================
# 6. 合并并保存
# =====================================================
df_final = pd.concat([df_1819, nearest_stations_info], axis=1)

df_final.to_csv(OUTPUT_PATH, index=False)

print("🎉 完成！结果已保存：", OUTPUT_PATH)
print(df_final.head())

✅ 1819 数据读取成功，共 2106 条
✅ 气象站点读取成功，共 129657 个站点
🚀 开始匹配最近的 5 个气象站（可能需要几分钟）...
🎉 完成！结果已保存： 1819_with_10_nearest_stations.csv
       UID      PHYLUM         CLASS           ORDER           FAMILY  \
0  2014737  ARTHROPODA  MALACOSTRACA        DECAPODA       CAMBARIDAE   
1  2012488  ARTHROPODA       INSECTA         DIPTERA        TABANIDAE   
2  2012489  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
3  2012490  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
4  2012491  ARTHROPODA     ARACHNIDA  TROMBIDIFORMES  TORRENTICOLIDAE   

          GENUS  TARGET_TAXON  TAXA_ID  TOTAL300  IS_DISTINCT300  ...  \
0      FAXONIUS      FAXONIUS     5131       1.0             1.0  ...   
1           NaN     TABANIDAE     4340       1.0             1.0  ...   
2           NaN  CHIRONOMIDAE     3581       2.0             0.0  ...   
3           NaN  CHIRONOMIDAE     3581       NaN             NaN  ...   
4  TORRENTICOLA  TORRENTICOLA     4371       NaN             NaN  ...   

